In [34]:
%%writefile practice10_2.cu
// Оптимизация доступа к памяти на GPU (CUDA)
// 1. реализовать две версии ядра:
//  a. с эффективным (коалесцированным) доступом к глобальной памяти;
//  b. с неэффективным доступом к памяти;
// 2. измерить время выполнения с использованием cudaEvent;
// 3. провести оптимизацию за счёт:
//  a. использования разделяемой памяти;
//  b. изменения организации потоков;
// 4. сравнить результаты и сделать выводы о влиянии доступа к памяти на  производительность GPU
#include <iostream>            // Подключение библиотеки для ввода-вывода (std::cout)
#include <cuda_runtime.h>      // Подключение CUDA Runtime API для работы с GPU

#define N (1 << 20)            // Размер массива: 2^20 = 1 048 576 элементов (~1 млн)
#define BLOCK_SIZE 256         // Размер блока потоков CUDA (обычно кратно warp size = 32)

// Коалесцированный доступ
__global__ void coalesced_access(float* data) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;  // Вычисление глобального индекса потока
    if (idx < N)                                      // Проверка выхода за пределы массива
        data[idx] = data[idx] * 2.0f;                // Умножение элемента на 2 (вычисление)
}

// Некоалесцированный доступ
__global__ void non_coalesced_access(float* data, int stride) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;  // Глобальный индекс потока
    int access_idx = (idx * stride) % N;             // "Разбросанный" индекс для некоалесцированного доступа
    data[access_idx] = data[access_idx] * 2.0f;     // Умножение элемента на 2
}

// Доступ с использованием shared memory
__global__ void shared_memory_access(float* data) {
    __shared__ float cache[BLOCK_SIZE];              // Выделение разделяемой памяти блока (быстрая память GPU)

    int idx = blockIdx.x * blockDim.x + threadIdx.x; // Глобальный индекс потока

    if (idx < N)
        cache[threadIdx.x] = data[idx];             // Копирование данных из глобальной памяти в shared memory

    __syncthreads();                                // Синхронизация потоков блока перед вычислением

    if (idx < N)
        data[idx] = cache[threadIdx.x] * 2.0f;      // Вычисление в shared memory и запись обратно в глобальную память
}

//  Функция измерения времени ядра с параметром stride
float measure_kernel(void (*kernel)(float*, int), float* d_data, int blocks, int stride=1) {
    cudaEvent_t start, stop;                        // Создание событий CUDA для замера времени
    cudaEventCreate(&start);                        // Инициализация события старта
    cudaEventCreate(&stop);                         // Инициализация события окончания
    cudaEventRecord(start);                         // Отметка начала времени
    kernel<<<blocks, BLOCK_SIZE>>>(d_data, stride); // Запуск ядра с указанным количеством блоков и потоков
    cudaEventRecord(stop);                          // Отметка окончания выполнения
    cudaEventSynchronize(stop);                     // Ожидание завершения всех потоков GPU
    float ms = 0;
    cudaEventElapsedTime(&ms, start, stop);         // Вычисление прошедшего времени в миллисекундах
    cudaEventDestroy(start);                        // Удаление события старта
    cudaEventDestroy(stop);                         // Удаление события окончания
    return ms;                                      // Возврат измеренного времени
}

//  Функция измерения времени ядра без stride (для coalesced/shared memory)
float measure_kernel_simple(void (*kernel)(float*), float* d_data, int blocks) {
    cudaEvent_t start, stop;                        // Создание событий CUDA для замера времени
    cudaEventCreate(&start);                        // Инициализация события старта
    cudaEventCreate(&stop);                         // Инициализация события окончания
    cudaEventRecord(start);                         // Отметка начала времени
    kernel<<<blocks, BLOCK_SIZE>>>(d_data);         // Запуск ядра без stride
    cudaEventRecord(stop);                          // Отметка окончания выполнения
    cudaEventSynchronize(stop);                     // Ожидание завершения всех потоков GPU
    float ms = 0;
    cudaEventElapsedTime(&ms, start, stop);         // Вычисление прошедшего времени в миллисекундах
    cudaEventDestroy(start);                        // Удаление события старта
    cudaEventDestroy(stop);                         // Удаление события окончания
    return ms;                                      // Возврат измеренного времени
}

int main() {             // Основная функция
    size_t size = N * sizeof(float);                // Вычисление размера массива в байтах
    float* h_data = new float[N];                   // Выделение памяти на хосте (CPU)
    for (int i = 0; i < N; i++) h_data[i] = 1.0f;  // Инициализация массива единицами

    float* d_data;                                  // Указатель на память GPU
    cudaMalloc(&d_data, size);                      // Выделение памяти на GPU
    int blocks = (N + BLOCK_SIZE - 1) / BLOCK_SIZE; // Вычисление числа блоков для покрытия всех элементов

    cudaMemcpy(d_data, h_data, size, cudaMemcpyHostToDevice);               // Копирование данных с CPU на GPU
    float t1 = measure_kernel_simple(coalesced_access, d_data, blocks);    // Замер времени для коалесцированного доступа

    cudaMemcpy(d_data, h_data, size, cudaMemcpyHostToDevice);               // Сброс данных на GPU
    float t2 = measure_kernel(non_coalesced_access, d_data, blocks, 32);   // Замер времени для некоалесцированного доступа

    cudaMemcpy(d_data, h_data, size, cudaMemcpyHostToDevice);               // Сброс данных на GPU
    float t3 = measure_kernel_simple(shared_memory_access, d_data, blocks); // Замер времени для shared memory

    std::cout << "Коалесцированный доступ к глобальной памяти: " << t1 << " ms" << std::endl;   // Вывод результата
    std::cout << "Некоалесцированный доступ к глобальной памяти: " << t2 << " ms" << std::endl; // Вывод результата
    std::cout << "Использование разделяемой памяти (shared memory): " << t3 << " ms" << std::endl; // Вывод результата

    cudaFree(d_data);  // Освобождение памяти на GPU
    delete[] h_data;   // Освобождение памяти на CPU
    return 0;          // Завершение программы
}



Overwriting practice10_2.cu


In [35]:

# Компиляция
!nvcc practice10_2.cu -o practice10_2 -arch=sm_75 -std=c++11            # -arch=sm_75  - архитектура GPU (Tesla T4 в Colab = sm_75)
                                                                    # -std=c++11 — стандарт C++
# Запуск
!./practice10_2

Коалесцированный доступ к глобальной памяти: 0.120576 ms
Некоалесцированный доступ к глобальной памяти: 0.278784 ms
Использование разделяемой памяти (shared memory): 0.043168 ms
